In [ ]:
!pip install -q transformers peft trl datasets bitsandbytes accelerate

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

MODEL_NAME = "Qwen/Qwen3-4B"
MAX_SEQ_LENGTH = 4096
LORA_RANK = 8

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.padding_side = "right"

enable_thinking = False

In [ ]:
lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.gradient_checkpointing_enable()
model.print_trainable_parameters()

In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.read_json("../data/preprocessed/dataset.jsonl", lines=True)

print(f"Samples: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head()

In [ ]:
def format_chat(row):
    messages = [
        {"role": "system", "content": row["system_prompt"]},
        {"role": "user", "content": row["prompt"]},
        {"role": "assistant", "content": row["ideal_response"]},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False, enable_thinking=enable_thinking)


df["text"] = df.apply(format_chat, axis=1)
dataset = Dataset.from_pandas(df[["text"]])

print(f"Example length (chars): {len(dataset[0]['text'])}")
print(dataset[0]["text"][:500])

In [ ]:
lengths = [len(tokenizer.encode(t)) for t in df["text"]]
print(f"Min: {min(lengths)}, Max: {max(lengths)}, Mean: {sum(lengths)//len(lengths)}")
print(f"MAX_SEQ_LENGTH: {MAX_SEQ_LENGTH}")
print(f"Примеров влезающих полностью: {sum(1 for l in lengths if l <= MAX_SEQ_LENGTH)}/{len(lengths)}")

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,
        packing=True,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        warmup_steps=10,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=5,
        optim="adamw_8bit",
        seed=42,
        output_dir="outputs",
        gradient_checkpointing=True,
    ),
)

In [ ]:
trainer.train()

## Проверка

In [ ]:
import gc
gc.collect()
torch.cuda.empty_cache()

model.eval()
model.config.use_cache = True

test_prompt = df["prompt"].iloc[0]
test_system = df["system_prompt"].iloc[0]

messages = [
    {"role": "system", "content": test_system},
    {"role": "user", "content": test_prompt},
]
inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_dict=True, return_tensors="pt", enable_thinking=enable_thinking).to(model.device)
print(f"Total input tokens: {inputs['input_ids'].shape[1]}")

with torch.no_grad(), torch.autocast("cuda"):
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.3, do_sample=True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

## Сохранение

Сохраняем LoRA-адаптер и (опционально) merged-модель в GGUF для Ollama.

In [ ]:
model.save_pretrained("koyash-qwen3-4b-lora")
tokenizer.save_pretrained("koyash-qwen3-4b-lora")

In [ ]:
from peft import AutoPeftModelForCausalLM

merged_model = AutoPeftModelForCausalLM.from_pretrained(
    "koyash-qwen3-4b-lora",
    torch_dtype=torch.float16,
    device_map="auto",
)
merged_model = merged_model.merge_and_unload()
merged_model.save_pretrained("koyash-qwen3-4b-merged")
tokenizer.save_pretrained("koyash-qwen3-4b-merged")

## Конвертация в GGUF и загрузка в Ollama

```bash
# Клонируем llama.cpp и конвертируем
git clone https://github.com/ggerganov/llama.cpp
pip install -r llama.cpp/requirements.txt

python llama.cpp/convert_hf_to_gguf.py koyash-qwen3-4b-merged --outfile koyash-qwen3-4b.gguf --outtype q4_k_m

# Создаём Modelfile и загружаем в Ollama
echo 'FROM ./koyash-qwen3-4b.gguf' > Modelfile
ollama create koyash-qwen3-4b-lora -f Modelfile
ollama run koyash-qwen3-4b-lora
```